# 03 — Hyperparameter Tuning & Overfitting Check
GridSearchCV with StratifiedKFold(5) to find best params, then train/test F1 comparison to detect overfitting.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd

from src.data import load_data, split_data
from src.tuning import tune_decision_tree, tune_knn, tune_naive_bayes
from src.models import train_decision_tree, train_knn, train_naive_bayes
from src.evaluate import evaluate

In [ ]:
X, y = load_data('../data/heart_failure_clinical_records_dataset.csv')
X_train, X_test, y_train, y_test = split_data(X, y)

## Grid Search Results

In [ ]:
dt_params, dt_cv_score = tune_decision_tree(X_train, y_train)
knn_params, knn_cv_score = tune_knn(X_train, y_train)
nb_params, nb_cv_score = tune_naive_bayes(X_train, y_train)

tuning_summary = pd.DataFrame([
    {'Model': 'Decision Tree', 'Best Params': str(dt_params), 'CV F1 (weighted)': round(dt_cv_score, 4)},
    {'Model': 'KNN',           'Best Params': str(knn_params), 'CV F1 (weighted)': round(knn_cv_score, 4)},
    {'Model': 'Naive Bayes',   'Best Params': str(nb_params), 'CV F1 (weighted)': round(nb_cv_score, 4)},
]).set_index('Model')

tuning_summary

## Overfitting Check — Train F1 vs Test F1
A large gap between train and test F1 indicates overfitting.

In [ ]:
rows = []

for train_fn, name in [
    (train_decision_tree, 'Decision Tree'),
    (train_knn,           'KNN'),
    (train_naive_bayes,   'Naive Bayes'),
]:
    pipe = train_fn(X_train, y_train)
    train_results = evaluate(pipe, X_train, y_train, model_name=f'{name} [train]')
    test_results  = evaluate(pipe, X_test,  y_test,  model_name=f'{name} [test]')
    gap = round(train_results['f1'] - test_results['f1'], 4)
    rows.append({
        'Model':      name,
        'Train F1':   round(train_results['f1'], 4),
        'Test F1':    round(test_results['f1'],  4),
        'Gap':        gap,
        'Overfit?':   'yes' if gap > 0.10 else 'no',
    })
    print()

In [ ]:
overfit_df = pd.DataFrame(rows).set_index('Model')
overfit_df